In [24]:
import immrax as irx
import jax
import jax.numpy as jnp


In [25]:
import os, sys
sys.path.append("/home/user/output_feedback/nhholyap/examples/")
from functools import partial
from jax import jit, lax, vmap
import matplotlib.pyplot as plt
from faulty_car.interval_functions import overlap_size_lax, overlap_size, overlap_sum_lax, overlap_size_lax_scaled
import time, copy
from typing import Optional, Dict, Any, Tuple, List
from datetime import datetime
from matplotlib.backends.backend_pdf import PdfPages

In [26]:
class FaultyNonHolonomicCarSI(irx.system.System):
    p:float
    """
    System Model of a "Faulty" Nonholonomic car.
    Dynamics:
    ---
    \dot{
        p_x,     v * cos(phi)     0   
        p_y,  =  v * sin(phi)   + 0
        phi,     0                omega
        v        0                a
        }
    ---
    """

    def __init__(self) -> None:
        # Tells immrax that the system is continuous
        self.evolution = 'continuous'
        # Tells immrax the number of states
        self.xlen = 3  #px, py, phi
        self.ulen = 2  #omega, v
        self.wlen = 1  #disturbance
        self.vlen = 3  #noise in output
        self.plen = 1

    def f(self, t: float, x: jax.Array, u: jax.Array, p: jax.Array) -> jax.Array:
        # assert x.shape == (self.xlen,), f"Expected x to be of shape ({self.xlen},), got {x.shape}"
        # assert u.shape == (self.ulen,), f"Expected u to be of shape ({self.ulen},), got {u.shape}"
        # assert w.shape == (self.wlen,), f"Expected w to be of shape ({self.wlen},), got {w.shape}"
        # assert p.shape == (self.plen,), f"Expected p to be of shape ({self.plen},), got {p.shape}"
        return jnp.array([
            u[0] * jnp.cos(x[2]),
            u[0] * jnp.sin(x[2]),
            p[0] * u[1]
        ])
    
    def h(self, t: float, x: jax.Array, v: jax.Array) -> jax.Array:
        """
        Definition of Observer.
        y = Cx + v
        Only px, py and steering angle can be observed.
        """
        return jnp.array([
            x[0] + v[0],
            x[1] + v[1],
        ])

In [27]:
from typing import Union
car_si_embsys = irx.natemb(FaultyNonHolonomicCarSI())
def propagate_interval_euler_SI(
        x0_interval: irx.Interval, 
        u_interval: Union[jnp.ndarray, irx.Interval], 
        p_interval: Union[jnp.ndarray, irx.Interval], 
        dt: float
    ) -> irx.Interval:
    xt_ut =  car_si_embsys.f(0., irx.i2ut(x0_interval), u_interval, p_interval) * dt + irx.i2ut(x0_interval)
    return irx.ut2i(xt_ut)

In [28]:
def loss_ff_generalized_scan_vmap_jax(u_ol, x_interval, p_nominal, p_faults, dt, num_sim_steps=10):
    # (This function is identical to the one in the previous answer)
    # 1. Propagate nominal case with scan
    def nominal_scan_fn(carry_x_ivl, _):
        next_x_ivl = propagate_interval_euler_SI(carry_x_ivl, u_ol, p_nominal, dt=dt)
        return next_x_ivl, None
    x_ivl_nominal, _ = jax.lax.scan(nominal_scan_fn, x_interval, None, length=num_sim_steps)

    # 2. Propagate all fault cases
    def propagate_single_fault(p_fault):
        def fault_scan_fn(carry_x_ivl, _):
            next_x_ivl = propagate_interval_euler_admire(carry_x_ivl, u_ol, p_fault, dt=dt)
            return next_x_ivl, None
        final_x_ivl, _ = jax.lax.scan(fault_scan_fn, x_interval, None, length=num_sim_steps)
        return final_x_ivl
    x_ivls_fault = jax.lax.map(propagate_single_fault, p_faults)

    # 3. Combine results
    all_ivls = jax.tree_util.tree_map(
        lambda n, f: jnp.concatenate([jnp.expand_dims(n, axis=0), f], axis=0),
        x_ivl_nominal, x_ivls_fault
    )

    # 4. Calculate pairwise loss with vmap
    num_faults = jax.tree_util.tree_leaves(p_faults)[0].shape[0]
    num_scenarios = 1 + num_faults
    i_indices, j_indices = jnp.triu_indices(num_scenarios, k=1)
    ivls_i = jax.tree_util.tree_map(lambda leaf: leaf[i_indices], all_ivls)
    ivls_j = jax.tree_util.tree_map(lambda leaf: leaf[j_indices], all_ivls)
    pairwise_losses = jax.vmap(overlap_size_lax, in_axes=(0, 0))(ivls_i, ivls_j)
    
    return jnp.sum(pairwise_losses)

In [29]:
def loss_ff_jax(u_ol, x_interval, dt, p_nominal, p_actuator_fault, observer_offset, num_steps=10):
    """
    Loss function: size of the overlap between nominal and faulty propagated intervals.
    This function is JIT-compatible.
    """
    # Define the single-step propagation for the nominal case
    def nominal_step_fn(i, x_propagated):
        return propagate_interval_euler_SI(x_propagated, u_ol, p_nominal, dt)

    # Define the single-step propagation for the faulty case
    def actuator_fault_step_fn(i, x_propagated):
        return propagate_interval_euler_SI(x_propagated, u_ol, p_actuator_fault, dt)

    # Use fori_loop for efficient JIT-compilation of the propagation loop
    x_ivl_nominal = jax.lax.fori_loop(0, num_steps, nominal_step_fn, x_interval)
    x_ivl_actuator_fault = jax.lax.fori_loop(0, num_steps, actuator_fault_step_fn, x_interval)

    return overlap_size_lax(x_ivl_nominal[:2], x_ivl_actuator_fault[:2])


In [30]:
def loss_ff_staged(u_ol, x_interval, dt, p_nominal, p_actuator_fault, observer_offset, num_steps=10, num_stages=2):
    """
    Loss function for staged inputs: size of the overlap between nominal and faulty propagated intervals.
    This function is JIT-compatible.

    Args:
        u_ol (jax.numpy.ndarray): The open-loop control inputs for all stages.
                                   Expected shape: (num_stages, control_dimension).
        x_interval (Interval): The initial state interval.
        dt (float): Time step.
        p_nominal (jax.numpy.ndarray): Parameters for the nominal system.
        p_actuator_fault (jax.numpy.ndarray): Parameters for the system with an actuator fault.
        observer_offset (jax.numpy.ndarray): The observer fault offset.
        num_steps (int): Total number of simulation steps. Must be divisible by num_stages.
        num_stages (int): The number of control stages.
    """
    # --- MODIFICATION START ---

    # 1. Calculate the number of steps per stage.
    # For JIT-compatibility, we assume num_steps is perfectly divisible by num_stages.
    # It's good practice to add a check outside of the JIT-ted function.
    num_steps_per_stage = num_steps // num_stages

    # 2. Define a generic, reusable step function generator.
    # This creates a propagation function for a given set of system parameters `p`.
    def create_step_fn(p):
        def step_fn(i, x_propagated):
            # Determine the current stage based on the simulation step `i`.
            stage_index = i // num_steps_per_stage
            
            # Select the control input for the current stage from the input array.
            current_u = u_ol[stage_index]
            
            # Propagate the system for one step with the selected control.
            return propagate_interval_euler_SI(x_propagated, current_u, p, dt)
        return step_fn

    # 3. Create the specific step functions for nominal and faulty cases.
    nominal_step_fn = create_step_fn(p_nominal)
    actuator_fault_step_fn = create_step_fn(p_actuator_fault)
    
    # --- MODIFICATION END ---

    # The fori_loop structure remains the same, but it now calls the new step functions
    # that handle the staged control selection internally.
    x_ivl_nominal = jax.lax.fori_loop(0, num_steps, nominal_step_fn, x_interval)
    x_ivl_actuator_fault = jax.lax.fori_loop(0, num_steps, actuator_fault_step_fn, x_interval)
    
    # The final loss calculation is unchanged.
    return overlap_size_lax(x_ivl_nominal[:2], x_ivl_actuator_fault[:2])

In [31]:
jax_grad_multiple = jax.grad(loss_ff_generalized_scan_vmap_jax, argnums=0)

def calculate_optimal_open_loop_multiple_faults(x_interval, learning_rate, u_initial, p_nominal, p_faults, dt, num_gd_steps=10, num_steps=100):
    # # Define the body of the gradient descent loop
    p_faults_stacked = jax.tree_util.tree_map(lambda *leaves: jnp.stack(leaves), *p_faults)
    def gradient_step(i, u_current):
        grad = jax_grad_multiple(u_current, x_interval, p_nominal, p_faults_stacked, dt, num_steps=num_steps)
        return u_current - learning_rate * grad
    

    u_opt = jax.lax.fori_loop(0, num_gd_steps, gradient_step, u_initial)

    return u_opt#u_initial#u_opt

In [61]:
loss_grad = jax.grad(loss_ff_jax, argnums=0)
def calculate_optimal_open_loop_u(x_interval, learning_rate, p_nominal, p_actuator_fault, observer_offset, dt, num_steps):
    """
    Calculates the optimal open-loop control input `u` using a multi-start gradient descent.
    This entire function is designed to be JIT-compiled.
    """
    num_gd_steps = 10
        
    # subkey, prng_key = jax.random.split(prng_key)
    # Initialize control input `u` with a random guess
    u_initial = jnp.ones(2) * .1

    # # Define the body of the gradient descent loop
    def gradient_step(i, u_current):
        grad = loss_grad(u_current, x_interval, dt, p_nominal, p_actuator_fault, observer_offset, num_steps,)
        return u_current - learning_rate * grad
    

    # # Use fori_loop for the gradient descent steps. This is crucial for JIT-compilation.
    u_opt = jax.lax.fori_loop(0, num_gd_steps, gradient_step, u_initial)


    return u_opt
loss_staged_grad = jax.grad(loss_ff_staged, argnums=0)
def calculate_optimal_open_loop_u_staged(x_interval, learning_rate, u_initial, p_nominal, p_actuator_fault, observer_offset, dt, num_steps):
    """
    Calculates the optimal open-loop control input `u` using a multi-start gradient descent.
    This entire function is designed to be JIT-compiled.
    """
    num_gd_steps = 10
        
    # subkey, prng_key = jax.random.split(prng_key)
    # Initialize control input `u` with a random guess
    

    # # Define the body of the gradient descent loop
    def gradient_step(i, u_current):
        grad = loss_staged_grad(u_current, x_interval, dt, p_nominal, p_actuator_fault, observer_offset, num_steps, num_stages=2)
        return u_current - learning_rate * grad
    

    # # Use fori_loop for the gradient descent steps. This is crucial for JIT-compilation.
    u_opt = jax.lax.fori_loop(0, num_gd_steps, gradient_step, u_initial)


    return u_opt


In [53]:
p_nominal_ivl = irx.icentpert(jnp.array([1.]), jnp.array([0.]))
p_actuator_fault_ivl = irx.interval(jnp.array([0.]), jnp.array([.25]))
observer_offset = jnp.ones(3) * .2
dt = 0.1
num_steps = 50
x0_interval = irx.icentpert(jnp.array([.1, .1, .0]), jnp.array([.1, .1, .1]))

In [62]:
jit_calculate_optimal_open_loop_u = jit(partial(
    calculate_optimal_open_loop_u,
    p_nominal=p_nominal_ivl,
    p_actuator_fault=p_actuator_fault_ivl,
    observer_offset=observer_offset,
    dt=dt,
    num_steps=num_steps
))

In [55]:
u_ol = jit_calculate_optimal_open_loop_u(
    x0_interval,
    10
)

In [63]:
jit_calculate_optimal_open_loop_u_staged = jit(partial(
    calculate_optimal_open_loop_u_staged,
    p_nominal=p_nominal_ivl,
    p_actuator_fault=p_actuator_fault_ivl,
    observer_offset=observer_offset,
    dt=dt,
    num_steps=num_steps
))

In [64]:
u_initial = jnp.array([
        [0., 0.1],
        [0.2, 0.]
    ])
u_ol = jit_calculate_optimal_open_loop_u_staged(
    x0_interval,
    1,
    u_initial
)

In [84]:
u_initial = jnp.array([
        [0., 0.1],
        [0.05, -0.05]
    ])
u_ol = jit_calculate_optimal_open_loop_u_staged(
    x0_interval,
    .5,
    u_initial
)

In [85]:
print(u_ol)

[[0.09971058 0.32818648]
 [0.1877347  0.04893967]]


In [86]:
loss_ff_staged(
    u_ol,
    x0_interval,
    dt,
    p_nominal_ivl,
    p_actuator_fault_ivl,
    observer_offset,
    num_steps=num_steps
)

Array(0., dtype=float32, weak_type=True)

In [101]:
import numpy as np
def get_noisy_measurement(x_actual, delta, key):
    offset = jnp.full_like(x_actual, delta / 2.0)
    return irx.Interval(x_actual - offset, x_actual + offset), key

# --- Main Simulation Function (with modifications for .npz saving) ---

def simulate_and_save_staged_trajectory_si(
    x0_interval: irx.Interval,
    u_staged: jax.Array,
    p_nominal: irx.Interval,
    p_faulty: irx.Interval,
    p_actual: jax.Array,
    dt: float,
    num_steps: int,
    num_stages: int,
    observer_offset: jax.Array = jnp.zeros(3),
    output_filename: str = "trajectory_history_si.npz", # <-- Changed default filename
    rng_key=jax.random.PRNGKey(0)
):
    """
    Simulates multiple system hypotheses with a staged open-loop control,
    logs the history, and saves it to a compressed .npz file.
    """
    # 1. & 2. Initialize Systems, States, and History (Unchanged)
    car_system = FaultyNonHolonomicCarSI()
    x_actual = (x0_interval.upper + x0_interval.lower) / 2.
    x_interval_nominal = copy.deepcopy(x0_interval)
    x_interval_faulty = copy.deepcopy(x0_interval)
    x_interval_observer_fault = x_interval_nominal + observer_offset
    history = {
        "nominal": {"lower": [x_interval_nominal.lower], "upper": [x_interval_nominal.upper]},
        "faulty": {"lower": [x_interval_faulty.lower], "upper": [x_interval_faulty.upper]},
        "observer_fault": {"lower": [x_interval_observer_fault.lower], "upper": [x_interval_observer_fault.upper]},
        "measurements": {"lower": [x0_interval.lower], "upper": [x0_interval.upper]},
        "actual_trajectory": [x_actual],
        "controls": []
    }

    # 3. Main Simulation Loop (Unchanged)
    if num_steps % num_stages != 0:
        raise ValueError("num_steps must be divisible by num_stages.")
    num_steps_per_stage = num_steps // num_stages

    for i in range(num_steps):
        stage_index = i // num_steps_per_stage
        u = u_staged[stage_index]
        x_interval_nominal = propagate_interval_euler_SI(x_interval_nominal, u, p_nominal, dt)
        x_interval_faulty = propagate_interval_euler_SI(x_interval_faulty, u, p_faulty, dt)
        x_interval_observer_fault = x_interval_nominal + observer_offset
        dx_actual = car_system.f(0., history["actual_trajectory"][-1], u, p_actual)
        x_actual = history["actual_trajectory"][-1] + dt * dx_actual
        measurement_interval, rng_key = get_noisy_measurement(x_actual, delta=0.1, key=rng_key)
        
        # Append JAX arrays to history
        history["nominal"]["lower"].append(x_interval_nominal.lower)
        history["nominal"]["upper"].append(x_interval_nominal.upper)
        history["faulty"]["lower"].append(x_interval_faulty.lower)
        history["faulty"]["upper"].append(x_interval_faulty.upper)
        history["observer_fault"]["lower"].append(x_interval_observer_fault.lower)
        history["observer_fault"]["upper"].append(x_interval_observer_fault.upper)
        history["measurements"]["lower"].append(measurement_interval.lower)
        history["measurements"]["upper"].append(measurement_interval.upper)
        history["actual_trajectory"].append(x_actual)
        history["controls"].append(u)

    # --- MODIFICATION START ---
    # 4. Prepare Data for NumPy and Save to .npz File
    print("\nConverting history to NumPy arrays for .npz file saving...")
    
    # Create a new, flat dictionary to hold the data formatted for saving.
    # We convert lists of vectors into 2D NumPy arrays and create unique keys.
    numpy_data = {
        'actual_trajectory': np.stack(history['actual_trajectory']),
        'controls': np.stack(history['controls'])
    }
    
    for fault_type in ["nominal", "faulty", "observer_fault", "measurements"]:
        numpy_data[f"{fault_type}_lower"] = np.stack(history[fault_type]["lower"])
        numpy_data[f"{fault_type}_upper"] = np.stack(history[fault_type]["upper"])
        
    # 5. Save to .npz File using NumPy
    try:
        # We use dictionary unpacking (**) to pass our data as keyword arguments.
        # This makes each key in `numpy_data` a named array in the .npz file.
        np.savez_compressed(output_filename, **numpy_data)
        print(f"Successfully saved trajectory history to {output_filename}")
    except Exception as e:
        print(f"\nError saving data to {output_filename}: {e}")
    # --- MODIFICATION END ---

    print("\nSimulation finished.")
    return history

In [94]:
p_actuator_fault_ivl

[0.] <= x <= [0.25]

In [111]:
simulate_and_save_staged_trajectory_si(
    x0_interval=x0_interval,
    u_staged=u_ol,
    p_nominal=p_nominal_ivl,
    p_faulty=p_actuator_fault_ivl,
    p_actual=jnp.zeros(1),
    dt=dt,
    num_steps=num_steps,
    num_stages=2,
    observer_offset=observer_offset,
)


Converting history to a single dictionary of NumPy arrays...
Successfully saved trajectory history dictionary to trajectory_history_si.npy

Simulation finished.


{'nominal': {'lower': [Array([ 0. ,  0. , -0.1], dtype=float32),
   Array([ 0.00992124, -0.00099544, -0.06718135], dtype=float32),
   Array([ 0.01980448, -0.00166481, -0.0343627 ], dtype=float32),
   Array([ 0.02963907, -0.00200738, -0.00154405], dtype=float32),
   Array([ 0.03941442, -0.00202277,  0.0312746 ], dtype=float32),
   Array([ 0.04912   , -0.00171098,  0.06409325], dtype=float32),
   Array([ 0.05874536, -0.00107234,  0.09691189], dtype=float32),
   Array([ 6.8280131e-02, -1.0753889e-04,  1.2973054e-01], dtype=float32),
   Array([0.07771404, 0.00118239, 0.16254918], dtype=float32),
   Array([0.08703694, 0.00279605, 0.19536783], dtype=float32),
   Array([0.09623878, 0.0047317 , 0.22818647], dtype=float32),
   Array([0.10530965, 0.00698727, 0.26100513], dtype=float32),
   Array([0.11423979, 0.00956032, 0.29382378], dtype=float32),
   Array([0.12301958, 0.01244808, 0.32664242], dtype=float32),
   Array([0.13163956, 0.01564744, 0.35946107], dtype=float32),
   Array([0.14009044, 0

In [110]:
def simulate_and_save_staged_trajectory_si(
    x0_interval: irx.Interval,
    u_staged: jax.Array,
    p_nominal: jax.Array,
    p_faulty: jax.Array,
    p_actual,
    dt: float,
    num_steps: int,
    num_stages: int,
    observer_offset: jax.Array = jnp.zeros(3),
    output_filename: str = "trajectory_history_si.npy", # <-- Changed default filename
):
    """
    Simulates multiple system hypotheses and saves the entire history
    as a single dictionary object in a .npy file.
    """
    # 1. & 2. & 3. Simulation logic is unchanged...
    car_system = FaultyNonHolonomicCarSI()
    x_actual = (x0_interval.upper + x0_interval.lower) / 2.
    x_interval_nominal = copy.deepcopy(x0_interval)
    x_interval_faulty = copy.deepcopy(x0_interval)
    history = {
        "nominal": {"lower": [x_interval_nominal.lower], "upper": [x_interval_nominal.upper]},
        "faulty": {"lower": [x_interval_faulty.lower], "upper": [x_interval_faulty.upper]},
        "observer_fault": {"lower": [], "upper": []},
        "actual_trajectory": [x_actual], "controls": []
    }
    history["observer_fault"]["lower"].append((x_interval_nominal + observer_offset).lower)
    history["observer_fault"]["upper"].append((x_interval_nominal + observer_offset).upper)
    num_steps_per_stage = num_steps // num_stages
    for i in range(num_steps):
        stage_index = i // num_steps_per_stage
        u = u_staged[stage_index]
        x_interval_nominal = propagate_interval_euler_SI(x_interval_nominal, u, p_nominal, dt)
        x_interval_faulty = propagate_interval_euler_SI(x_interval_faulty, u, p_faulty, dt)
        x_interval_observer_fault = x_interval_nominal + observer_offset
        dx_actual = car_system.f(0., history["actual_trajectory"][-1], u, p_actual)
        x_actual = history["actual_trajectory"][-1] + dt * dx_actual
        # Append JAX arrays to history...
        history["nominal"]["lower"].append(x_interval_nominal.lower)
        history["nominal"]["upper"].append(x_interval_nominal.upper)
        history["faulty"]["lower"].append(x_interval_faulty.lower)
        history["faulty"]["upper"].append(x_interval_faulty.upper)
        history["observer_fault"]["lower"].append(x_interval_observer_fault.lower)
        history["observer_fault"]["upper"].append(x_interval_observer_fault.upper)
        history["actual_trajectory"].append(x_actual)
        history["controls"].append(u)

    # --- MODIFICATION START ---
    # 4. Prepare Data for Saving as a Single Dictionary
    print("\nConverting history to a single dictionary of NumPy arrays...")
    
    # We will save this single dictionary object.
    data_to_save = {
        'actual_trajectory': np.stack(history['actual_trajectory']),
        'controls': np.stack(history['controls'])
    }
    for fault_type in ["nominal", "faulty", "observer_fault"]:
        data_to_save[fault_type] = {
            "lower": np.stack(history[fault_type]["lower"]),
            "upper": np.stack(history[fault_type]["upper"])
        }
        
    # 5. Save the dictionary object to a .npy file.
    # We must set `allow_pickle=True` for this to work.
    try:
        np.save(output_filename, data_to_save, allow_pickle=True)
        print(f"Successfully saved trajectory history dictionary to {output_filename}")
    except Exception as e:
        print(f"\nError saving data to {output_filename}: {e}")
    # --- MODIFICATION END ---

    print("\nSimulation finished.")
    return history # Return the original JAX-based history